In [217]:
include("../src/main.jl");
println("MBh = $MBH")
println("MUnit = $M_unit")

MBh = 6.2e9
MUnit = 3.0e26


In [218]:
#dump_filepath = "../src/models/iharm3dDumps/dump_001.h5";
dump_filepath = "../../gpumonty/data/sample_dump_SANE_a+0.94_MKS_0900.h5"
#dump_filepath = "../../../../Downloads/torus.out0.00356.h5";

"../../gpumonty/data/sample_dump_SANE_a+0.94_MKS_0900.h5"

In [219]:
#TODO: put this in reading file
const N1 = 288
const N2 = 128
const N3 = 128

const METRIC = "MKS" #FMKS or MKS TODO: prob have to be read from file
const trat_large = 20. #TODO: prob have to be read from file
const trat_small = 1. #TODO: prob have to be read from file
const beta_crit = 1.0 #TODO: prob have to be read from file
const game = (4. /3.)  # Ion adiabatic index  TODO: prob have to be read from file
const gamp = (5. /3.)  # Electron adiabatic index TODO: prob have to be read from file
const gam = (1.444444)  # Total adiabatic index TODO: prob have to be read from file
const Ne_factor = 1.0  # Scaling factor for electron number density TODO: prob have to be read from file
const rmin_geo = 1.00187575798832   #TODO: Has to be read from file as Rin and compared to the value chosen
const rmax_geo = 100. #TODO: Has to be read from file as Rin and compared to the value chosen
const th_beg = 1.74e-2 #TODO: Idk where this comes from, check ipole source code
const sigma_cut = 1.0 #TODO: maybe put it somewhere else?
const sigma_cut_high = -1.0
const startx::MVec4 = [0, 1.635684465252566e-01, 0, 0]#TODO: prob have to be read from file
const stopx::MVec4 = [1,  6.907755278982137, 1, 2 * π]#TODO: prob have to be read from file
const dx::MVec4 =[0, 2.341731539047528e-02, 7.812500000000000e-03, 4.908738521234052e-02]
const bhspin = 0.9375 #TODO: prob have to be read from file
const hslope = 0.3 #TODO: prob have to be read from file


0.3

In [220]:
include("../src/main.jl");
const simulation_data = load_data(dump_filepath);

Loading data from '../../gpumonty/data/sample_dump_SANE_a+0.94_MKS_0900.h5' into 'iharm' module...
Primitives successfully loaded. Dimensions: (288, 128, 128)
Calculating physical quantities...
Using mixed tp_over_te with trat_small = 1.0, trat_large = 20.0, and beta_crit = 1.0
All primitives successfully loaded. Dimensions: (288, 128, 128)


In [221]:
#Analytic parameters

#Setting up the parameters
#Observer distance in Rg
const ro = 1000.0
#Observer inclination in degrees
const th = 163.0

#Observer azimuth in degrees
const phi = 0.0

# Number of pixels in the x and y direction. The number of geodesics calculated will be res^2
const res = 320
const pixels_x = 320
const pixels_y = 320
# Distance to the source in parsecs
const SourceD = 16.9e6 * PC
const Rout = 1000.0
const Rstop = 100.0
const Rh = 1 + sqrt(1. - bhspin * bhspin);

#Check if these are correct
#const cstartx = [0.0, log(Rh), 0.0, 0.0]#TODO: prob have to be read from file
const cstartx = MVec4(0.0, 0.0, 0.0, 0.0)#TODO: prob have to be read from file "This one is from mario's file"

const cstopx = MVec4(0.0, log(Rout), 1.0, 2.0 * π)#TODO: prob have to be read from file

# Frequency observed by the camera in Hz
const freq = 230e9;

# Size of the screen in Rg in both directions
const DXsize = SourceD/L_unit/MUAS_PER_RAD * 160
const DYsize = SourceD/L_unit/MUAS_PER_RAD * 160
# Observer fov in radians (this can be understood as size of the plane camera sees over the distance ro)
# This should be atan, but for small angles it is approximately equal to the angle itself
const fovx = DXsize/ro
const fovy = DYsize/ro
const xoff = 0.0
const yoff = 0.0


0.0

In [222]:
using ProgressMeter
include("../src/main.jl");

# Find camera in native coordinates
Xcamera = MVec4(camera_position(ro, th, phi, bhspin, Rout))

# Scales the intensity of each pixel by the real size of each pixel
scale_factor = CalculateScaleFactor(DXsize, DYsize, pixels_x, pixels_y, SourceD, L_unit)
println("scale_factor = $scale_factor")
const maxnstep = 15000
# Generate geodesics
println("Utilizing $(Threads.nthreads()) threads for geodesic calculation.")

p = Progress(
    pixels_x * pixels_y; 
    desc = "Raytracing Image...", 
    showspeed = true, 
    barlen = 30
)
ProgressMeter.ijulia_behavior(:clear)

freq_unitless = freq * HPL/(ME * CL * CL) 
Image = zeros(Float64, pixels_x, pixels_y)
@time begin
   Threads.@threads for i in 0:(pixels_x - 1)
        tid = Threads.threadid()
        for j in 0:(pixels_y - 1)
            traj = Vector{OfTraj}()
            sizehint!(traj, maxnstep)
            nstep = get_pixel(traj, i, j, Xcamera, maxnstep, fovx, fovy, freq_unitless, pixels_x, pixels_y, bhspin, Rh, Rout, Rstop, xoff, yoff) 
            
            resize!(traj,nstep)
            integrate_emission!(traj, nstep, Image, i + 1, j + 1, freq, bhspin, simulation_data)
            ProgressMeter.next!(
                p; 
                showvalues = [
                    (:thread_id, tid), 
                    (:pixel, "($i, $j)"), 
                    (:total_done, "$(i*pixels_y + j)/$(pixels_x * pixels_y)")
                ]
            )
        end
    end
    Image *= freq^3;
end
finish!(p);

Raytracing Image... 100%|██████████████████████████████| Time: 0:08:32 ( 5.01 ms/it)
    thread_id: 1
        pixel: (319, 319)
   total_done: 102399/102400


512.479300 seconds (5.03 G allocations: 322.546 GiB, 19.05% gc time, 0.25% compilation time)


In [224]:
OutputStokesParameters(Image, freq, scale_factor, res, SourceD)

Image processing complete. Calculating total flux and averages...
Scale = 5.876096595457265e-01
imax = 129, jmax = 154, Imax = 0.0009762617338604302, Iavg = 6.899054914013961e-6
Total Flux Fnu = 4.151246140632046e-01 Jy
nuLnu = 3.262802619967386e40


In [225]:
using DelimitedFiles

writedlm("./Image_POLCODECOMP_320.txt", Image)
